## 🗂️ 🌧️ Organization Metrics for Precipitation Structure and Synthetically Randomized Distributions 🌧️ 🗂️

From ORCESTRA/PICCOLO SEA-POL radar observations, identify precipitating entities and compute metrics of organization to describe their spatial structure. Calculate the average edge-to-edge distance between all possible entity pairs ($\overline{D}$), the Radar Organization Metric (ROME)[1], the Simple Convective Aggregation Index (SCAI)[2], $L_{org}$[4], and the index of organization ($I_{org}$)[4]. 

Compute synathetically randomized distributions of observed radar scenes. 

See references at the bottom of this notebook. 

SEA-POL data is available through IPFS on the ORCESTRA data browser at https://browser.orcestra-campaign.org/#/?s=. 

For more on data access with IPFS, see https://orcestra-campaign.org/ipfs.html#ipfs. 

### Import relevant libraries, access SEA-POL radar data, and process variables. 

In [1]:
import numpy as np
import xarray as xr
import netCDF4 as nc
import pandas as pd
from datetime import datetime
from scipy.stats import norm
from scipy.ndimage import gaussian_filter, label, generate_binary_structure 

In [2]:
# Access the PICCOLO_level4_rainrate_2D.zarr dataset at the following URL:
url = "http://127.0.0.1:8080/ipns/latest.orcestra-campaign.org/products/METEOR/SEA-POL/PICCOLO_level4_rainrate_2D.zarr/"

# Open the 2D dataset using xarray with the zarr engine. 
ds = xr.open_dataset(url, engine="zarr")
vars_list = list(ds.variables.keys())

# Print a list of available variables in the dataset.
print(vars_list)

['DBZ', 'HID', 'RAINRATE', 'X', 'Y', 'elevation_angle', 'grid_mapping', 'heading', 'latitude', 'longitude', 'start_time', 'stop_time', 'time']


In [3]:
# Name relevant variables and read them in. 
dbz = ds.variables['DBZ'][:]
print('done with dbz') # Print 'done' statements to track loading progess. 

rainrate = ds.variables['RAINRATE'][:]
print('done with rainrate')

x = ds.variables['X'][:]
print('done with x')

y = ds.variables['Y'][:]
print('done with y')

latitude = ds.variables['latitude'][:]
print('done with latitude')

longitude = ds.variables['longitude'][:]
print('done with longitude')

start_time = ds.variables['start_time'][:]
print('done with start_time')

stop_time = ds.variables['stop_time'][:]
print('done with stop_time')

time = ds.variables['time'][:]
print('done with time')

done with dbz
done with rainrate
done with x
done with y
done with latitude
done with longitude
done with start_time
done with stop_time
done with time


In [4]:
# Spatially subset radar data to within a specified radius from SEA-POL. 

# Re-name rainrate and reflectivity arrays for easier use. 
dbz = dbz[:, :, :]
rr = rainrate[:, :, :]

# Create 2D meshgrid of x and y. 
X, Y = np.meshgrid(x, y)

# Calculate the distance from the center for each grid point. 
center_x = float((x.max() + x.min()) / 2)
center_y = float((y.max() + y.min()) / 2)
distance_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)

# Create a mask for points within desired radius. 
radius_mask = distance_from_center <= 120000 # 120 km radius in meters


# Apply the mask to all rainrate arrays. 
rr_120km = np.where(radius_mask, rr, np.nan)

# Ensure that NaNs, -32768, and masked values ('--') in the original arrays are preserved in the masked arrays. 
rr_120km = np.where(np.isnan(rr) | (rr == -32768), np.nan, rr_120km)


# Apply the mask to all dbz arrays. 
dbz_120km = np.where(radius_mask, dbz, np.nan)

# Ensure that NaNs, -32768, and masked values ('--') in the original arrays are preserved in the masked arrays
dbz_120km = np.where(np.isnan(dbz) | (dbz == -32768), np.nan, dbz_120km)

### Radar Oragnization Metric (ROME) and $\overline{D}$

ROME was defined by Retsch et al. 2020[1] at https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2019JD031801. 

A data and script repository from Retsch et al. 2020[1] can be found at https://bridges.monash.edu/articles/dataset/Retsch_etal_2020/11987607. 

In [5]:
import scipy.spatial 
from itertools import combinations
from skimage import measure
from scipy.ndimage import binary_fill_holes
from scipy.spatial import ConvexHull
from scipy.spatial import cKDTree

In [6]:
def calc_ROME(data_array, latitude2d, longitude2d, data_min=-30, data_max=np.inf, nthreshold=2, includeextras=False):
    """
    Calculate the ROME metric for a given radar scene. 

    Arguments:
        data_array : 2D array of radar scene data (e.g., rain rate, reflectivity)
        latitude2d : 2D array of latitudes corresponding to the radar scene grid points
        longitude2d : 2D array of longitudes corresponding to the radar scene grid points
        data_min : Minimum data value threshold for identifying precipitating entities (default: -30 dBZ)
        data_max : Maximum data value threshold for identifying precipitating entities (default: infinity)
        nthreshold : Minimum number of entities required to calculate ROME (default: 2)
        includeextras : If True, return additional information about entities and pairs (default: False)

    Returns:
        rome: ROME metric for scene (units of km^2). 
        If includeextras is True, also returns:
            entity_boundaries: Dictionary containing boundary coordinates for each entity.
            distance_list: List of distances between all entity pairs.

    """

    # Create a data mask where pixels within the specified minimum and maximum range are given a 1 and all other pixels are 0. 
    data_mask = np.where(np.isnan(data_array), 0, np.where((data_array >= data_min) & (data_array < data_max), 1, 0))

    # Set up an 8-connectivity structure for identifying contiguous regions with a 1. 
    s = generate_binary_structure(2, 2) 
    
    # Use scipy's label function to identify contiguous regions in the binary mask. 
    # 'entity_mask' will have unique numerical labels for each entity. 
    # 'nentities' is the total number of entities found. 
    entity_mask, nentities = label(data_mask, structure=s)

    # calculate the area of each entity by counting the number of pixels in each labeled region (1 pixel = 1 km^2)
    entity_ids = np.unique(entity_mask[~np.isnan(entity_mask)])
    entity_ids = entity_ids[entity_ids != 0]
    entity_areas = [(entity_mask == eid).sum() for eid in entity_ids]
    areas = entity_areas # save the area of each entity in a list 

    # Find all unqiue entity pairs. 
    entity_ids = range(1, nentities + 1)  # entity IDs start from 1 (0 is blank space)
    entity_pairs = list(combinations(entity_ids, 2)) # all unique pairs of entities


    entity_boundaries = {}
    entities = entity_mask


    # Loop through each entity to find its boundary coordinates.
    for i in range(1, nentities + 1):

        # Create a mask for the current entity. 
        entity_mask = (entities == i)

        # Fill holes in the entity mask. 
        filled_entity_mask = binary_fill_holes(entity_mask)

        # Find contours of the filled entity mask. 
        contours = measure.find_contours(filled_entity_mask, level=0.5)

        # Convert the contours to a list of coordinates.
        boundary_coordinates = []
        for contour in contours:
            for coord in contour:
                boundary_coordinates.append((int(coord[0]), int(coord[1])))

        # Store the boundary coordinates for the current entity.
        entity_boundaries[i] = boundary_coordinates

        # Convert to lat and lon for future plotting. 
        boundary_lat_lon = []
        nrows, ncols = latitude2d.shape
        for coord in boundary_coordinates:
            y_idx = int(round(coord[0]))
            x_idx = int(round(coord[1]))
            y_idx = np.clip(y_idx, 0, nrows - 1)
            x_idx = np.clip(x_idx, 0, ncols - 1)
            boundary_lat_lon.append((latitude2d[y_idx, x_idx], longitude2d[y_idx, x_idx]))

        # Check if boundary_lat_lon is not empty before unpacking. 
        if boundary_lat_lon:
            boundary_lat, boundary_lon = zip(*boundary_lat_lon)
        else:
            boundary_lat, boundary_lon = [], []

        # Save in the dictionary. 
        entity_boundaries[i] = {
            "boundary_cartesian": boundary_coordinates,
            "boundary_lat": boundary_lat,
            "boundary_lon": boundary_lon
        }

    # Save the scalar values for each unique entity pair in a list. 
    c_list = []
    distance_list = []

    if nentities >= nthreshold:
        for pair in entity_pairs:
            entity1, entity2 = pair

            # Find the area of each entity. 
            area1 = areas[entity1 - 1]  # subtract 1 to match index
            area2 = areas[entity2 - 1]

            # area_A = largest and area_B = smallest 
            area_A = max(area1, area2)
            area_B = min(area1, area2)

            # Calculate the distance between the edges of the entities. 
            boundary_1 = entity_boundaries[entity1]["boundary_cartesian"]
            boundary_2 = entity_boundaries[entity2]["boundary_cartesian"]

            # Calculate the distance between the two closest edges of the boundaries. 
            boundary_1_array = np.array(boundary_1)
            boundary_2_array = np.array(boundary_2)

            # Build a KDTree for one of the boundaries. 
            tree = cKDTree(boundary_2_array)

            # Query the nearest distances from boundary_1 to boundary_2. 
            distances, _ = tree.query(boundary_1_array)

            # Get the minimum distance. 
            distance = np.min(distances)
            distance_list.append(distance)

            # Calculate ROME index, c. 
            c = area_A + min(1, area_B / distance**2) * area_B
            c_list.append(c)

        # ROME is the avaerage of all c values for all unique entity pairs.
        rome = np.mean(c_list)
        
    else: 
        rome = np.nan

    if includeextras:
        return rome, entity_boundaries, distance_list
    else:
        return rome

In [65]:
# Need to read in entity count and average entity area here to use in the loop below.
# See precip_properties_geom notebook in the repository. 

########### READ IN N_LIST AND ABAR_LIST HERE ############

In [7]:
# Compute ROME using the function for each radar scan in the 120 km subset.

rome_list = []  
distances_list = [] # stores all edge-to-edge distances between entity pairs

# Loop over each 2D precipitation scene. 
for i in range(rr_120km.shape[0]):

    # if N = 1, ROME = Abar (see precip_properties_geom notebook)
    if n_list[i] == 1: 
        rome_list.append(abar_list[i])
        distances_list.append([np.nan])
        continue

    # if N = 0 or NaN, ROME = NaN
    if n_list[i] == 0 or np.isnan(n_list[i]):
        rome_list.append(np.nan)
        distances_list.append(np.nan)
        continue

    # if N > 1, calculate ROME using the function
    rome, entity_boundaries, distance_list = calc_ROME(dbz_120km[i], latitude[i], longitude[i], data_min=-30, data_max=np.inf, nthreshold=2, includeextras=True)
    rome_list.append(rome)
    distances_list.append(distance_list)


# Dbar is the average distance between entities
dbar_list = [np.nanmean(d) if isinstance(d, (list, np.ndarray)) and len(d) > 0 else np.nan for d in distances_list]

NameError: name 'n_list' is not defined

### Simple Convective Aggregation Index (SCAI)

SCAI was defined by Tobin et al. (2012)[2] at https://doi.org/10.1175/JCLI-D-11-00258.1. 

In [8]:
from scipy.ndimage import center_of_mass

In [9]:
def calc_SCAI(data_array, data_min=-30, data_max=np.inf, nthreshold=2, nmax = 14300, l = 240):
    """
    Calculate the SCAI metric for a given radar scene. 

    Arguments:
        data_array : 2D array of radar scene data (e.g., rain rate, reflectivity)
        data_min : Minimum data value threshold for identifying precipitating entities (default: -30 dBZ)
        data_max : Maximum data value threshold for identifying precipitating entities (default: infinity)
        nthreshold : Minimum number of entities required to calculate SCAI (default: 2)
        nmax: half of the total number of domain grid cells (default: 14300 for 120 km subset)
        l: Domain length (default: 240 km for 120 km subset)

    Returns:
        scai: SCAI metric for scene (unitless). 

    """

    # Create a data mask where pixels within the specified minimum and maximum range are given a 1 and all other pixels are 0. 
    data_mask = np.where(np.isnan(data_array), 0, np.where((data_array >= data_min) & (data_array < data_max), 1, 0))

    # Set up an 8-connectivity structure for identifying contiguous regions with a 1. 
    s = generate_binary_structure(2, 2) 
    
    # Use scipy's label function to identify contiguous regions in the binary mask. 
    # 'entity_mask' will have unique numerical labels for each entity. 
    # 'nentities' is the total number of entities found. 
    entity_mask, nentities = label(data_mask, structure=s)

    # Find all unique entity pairs. 
    entity_ids = np.unique(entity_mask[~np.isnan(entity_mask)])
    entity_ids = entity_ids[entity_ids != 0]
    entity_ids = range(1, nentities + 1)  # entity IDs start from 1 (0 is blank space)
    entity_pairs = list(combinations(entity_ids, 2)) # all unique pairs of entities

    # Save the scalar values for each unique entity pair in a list. 
    allpair_centroid_distances = []

    if nentities >= nthreshold:
        for pair in entity_pairs:
            entity1, entity2 = pair

            mask1 = (entity_mask == entity1)
            mask2 = (entity_mask == entity2)

            # Get centroids using center_of_mass.
            centroid1 = center_of_mass(mask1)
            centroid2 = center_of_mass(mask2)

            # Calculate Euclidean distance between centroids. 
            dist = np.sqrt((centroid1[0] - centroid2[0])**2 + (centroid1[1] - centroid2[1])**2)
            allpair_centroid_distances.append(dist)

        d_0 = np.exp(np.log(allpair_centroid_distances).sum() / len(entity_pairs))
        scai = (nentities / nmax) * (d_0 / l) * 1000

    else: 
        scai = np.nan

    return scai

In [10]:
# Compute and store SCAI values for each radar scan in the 120 km subset.
scai_values = []
for i in range(len(dbz_120km)):
    dbz_scene = dbz_120km[i]
    scai = calc_SCAI(dbz_scene)
    scai_values.append(scai)

KeyboardInterrupt: 

### $L_{org}$ and Synthetically Randomized Distributions

$L_{org}$ was defined by Biagioli and Tompkins 2023[3] at https://doi.org/10.1175/JAS-D-23-0103.1.  

A script repository from Biagioli and Tompkins 2023[3] can be found at https://github.com/giobiagioli/organization_indices. 

#### Libraries and Functions

In [11]:
from sklearn.neighbors import NearestNeighbors
from scipy.ndimage import measurements
from scipy.ndimage import center_of_mass
import matplotlib.pyplot as plt
from scipy.ndimage import generate_binary_structure
from scipy.ndimage import label
from scipy.interpolate import interp1d
import math as math
from scipy.integrate import cumulative_trapezoid
from scipy.ndimage import binary_erosion
from skimage.morphology import disk
from scipy.spatial import cKDTree

In [ ]:
# Define the domain size and bin sizes for searching for all neighbors. 
bins = np.arange(0, 242, 2) # can edit bins based on desired range and size 
domain_x = 240  # used when calculating metrics for the 120 km radius scans
domain_y = 240 
rmax = 240 # can change rmax based on desired maximum search distance for neighboring entities 
dxy = 1
start_idx = (491 - 241) // 2
end_idx = start_idx + 241

In [14]:
def get_scene_data(data_array, data_min=-30, data_max=np.inf):
    """
    Get the shape/mask and a list of precipitating entity areas for a given radar scene. 

    Arguments:
        data_array : 2D array of radar scene data (e.g., rain rate, reflectivity)
        data_min : Minimum data value threshold for identifying precipitating entities (default: -30 dBZ)
        data_max : Maximum data value threshold for identifying precipitating entities (default: infinity)

    Returns:
        scene_mask: 2D array of a blank scene of the same shape as the input data array. 
        entity_areas: List of areas (in km^2) for each identified precipitating entity.

    """

    # Define the entity mask based on the min and max thresholds, while properly handling NaNs and masked values.
    rain_mask = np.where(
        (np.ma.getmask(data_array)) | np.isnan(data_array) | (data_array == '--') | (data_array == -32768),
        np.nan,
        np.where((data_array >= data_min) & (data_array < data_max), 1, 0)
    )

    # Label each 8-connected structure. 
    s = generate_binary_structure(2, 2) 
    entity_mask, nentities = label(np.nan_to_num(rain_mask, nan=0), structure=s)

    # Get the centroids of each entity
    centroids = center_of_mass(data_array, entity_mask, range(1, nentities + 1))
    centroids = np.array(centroids)

    # Extra sanity check! 
    # Make sure all NaN and missing data flags are processed correctly. 
    bad_mask = (np.ma.getmask(data_array) | np.isnan(data_array) | (data_array == '--') | (data_array == -32768))
    cluster_mask = np.where(bad_mask, np.nan, entity_mask)

    # Get the shape of the current scene and generate a blank scene of the same shape
    # This put the "slices" of missing data in the same spot as in the real scene. 
    orig_mask = cluster_mask.mask if hasattr(cluster_mask, 'mask') else np.isnan(cluster_mask)
    orig_shape = cluster_mask.shape
    nan_seapol_mask = np.full((241, 241), np.nan)
    start_y = (241 - orig_shape[0]) // 2
    start_x = (241 - orig_shape[1]) // 2
    center_block = np.zeros(orig_shape, dtype=float)
    center_block[orig_mask] = np.nan
    center_block[~orig_mask] = 0
    nan_seapol_mask[start_y:start_y + orig_shape[0], start_x:start_x + orig_shape[1]] = center_block
    scene_mask = nan_seapol_mask.copy()

    # Get the list of entity count and area. 
    entity_ids = np.unique(cluster_mask[~np.isnan(cluster_mask)])
    entity_ids = entity_ids[entity_ids != 0]
    entity_areas = [(cluster_mask == eid).sum() for eid in entity_ids]

    if len(entity_areas) > 0:
        return scene_mask, entity_areas
    else:
        return scene_mask, []

In [15]:
def generate_random(nan_seapol_mask, entity_areas, n_threshold = 1, repeat_count = 100):
    """
    Generate synthetically randomized radar scenes with the same number and area of precipitating entities as the real radar scene, but with randomly placed entities.

    Arguments:
        nan_seapol_mask : 2D array of the shape/mask of the original radar scene 
        entity_areas : List of areas (in km^2) for each identified precipitating entity
        n_threshold : Minimum number of entities required to perform randomization (default: 1)
        repeat_count : Number of random scenes to generate (default: 100)

    Returns:
        random_scenes: List of 2D arrays representing the randomized radar scenes
    """

    # Create an empty list to store results. 
    random_scenes = []

    # Use the same number and areas as the real scene's entities. 
    areas = np.array(entity_areas)
    radii = np.sqrt(areas / np.pi).astype(int) # assume each entity is a circle and find its radius 
    num_circles = len(radii)

    # Only perform the randomization if there are more entities than the threshold. 
    if num_circles > n_threshold:

        # Precompute valid mask and eroded masks only once per unique radius. 
        valid_mask = ~np.isnan(nan_seapol_mask)
        unique_radii, inverse_indices = np.unique(radii, return_inverse=True)

        # Eroded masks make sure the centers of the circles are placed far enough from the edges to fit the entire circle within the valid area. 
        eroded_masks_dict = {r: binary_erosion(valid_mask, structure=disk(r)) for r in unique_radii}
        valid_center_coords_dict = {r: np.argwhere(eroded_masks_dict[r]) for r in unique_radii}

        for repeat in range(repeat_count):
            fakescene = np.zeros((241,241), dtype=float)
            placed_centers = []
            used_coords = set()
            for i, r in enumerate(radii):
                valid_centers = valid_center_coords_dict[r]
                if len(valid_centers) == 0:
                    continue
                # Shuffle possible locations for center placement.
                idxs = np.random.permutation(len(valid_centers))
                found = False
                for idx in idxs:
                    y, x = valid_centers[idx]
                    # Check distance to all previously placed circles (ensure non-overlapping). 
                    if all((y - cy) ** 2 + (x - cx) ** 2 >= (r + radii[j]) ** 2 for j, (cy, cx) in enumerate(placed_centers)):
                        placed_centers.append((y, x))
                        found = True
                        break
                if not found:
                    continue

            yy, xx = np.ogrid[:241, :241]
            for i, (cy, cx) in enumerate(placed_centers):
                mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radii[i] ** 2
                fakescene[mask] = 1 # set the cells within the area of the circle to 1 (indicating presence of the entity)

            fakescene[np.isnan(nan_seapol_mask)] = np.nan
            random_scenes.append(fakescene) # make sure nans are in the same place as original scene 


        return random_scenes

    else:
        return np.nan
    

In [16]:
def calculate_besag_for_scene(data_array, bins, data_min = 0, data_max = np.inf):
    """
    Calculate the observed Besag's L function for a real, observed radar scene. 

    Arguments:
        data_array : 2D array of radar scene data (e.g., rain rate, reflectivity)
        bins: 1D array of distance bins for calculating the cumulative count of neighbors within each distance bin
        data_min : Minimum data value threshold for identifying precipitating entities (default: -30 dBZ)
        data_max : Maximum data value threshold for identifying precipitating entities (default: infinity)

    Returns:
        Besag: 1D array representing Besag's L function for the observed radar scene
    """

    # Re-import label and center_of_mass here to avoid naming issues elsewhere in the notebook. 
    from scipy.ndimage import label, center_of_mass

    # Find the precipitating entities in the scene based on the min and max thresholds, while properly handling NaNs and masked values.
    data_mask = np.where(
        np.isnan(data_array),
        0,
        np.where((data_array > data_min) & (data_array <= data_max), 1, 0)
    )

    # Label each 8-connected structure (as done elsewhere throughout this notebook).
    s = generate_binary_structure(2, 2)
    entity_mask, nentities = label(data_mask, structure=s)

    # If there are fewer than 2 entities, Besag's L function cannot be calculated, so return an array of zeros.
    if nentities < 2:
        return np.zeros_like(bins, dtype=float)
    
    # Get the centroids of each entity. 
    centroids = center_of_mass(data_array, entity_mask, range(1,nentities+1))
    centroids = np.array(centroids)
    j9 = centroids.copy()

    # Create a list for finding the nearest neighbor distances for each entity. 
    NNdist = np.zeros(len(centroids))

    # Create an array to store the cumulative count of neighbors within each distance bin for each entity.
    cum_counting = np.zeros((len(centroids), len(bins)))
    ncnv = len(centroids)

    for k in range(len(centroids)):

        # Make sure there are at least 2 entities to compute distances. 
        if ncnv < 2:
            continue  
        
        # Create an empty histogram to store the count of neighbors within each distance bin for the current entity.
        hist = np.zeros(len(bins))
        extra_pts = np.delete(j9, list(range(k, j9.shape[0], len(centroids))), axis=0)

        # Find the nearest neighbors to centroid k. 
        tree = cKDTree(extra_pts)
        dist, ii = tree.query(centroids[k,:], ncnv-1)
        dist *= dxy

        # If k has only one neighbor, make sure dist and ii are arrays. 
        if np.isscalar(dist):
            dist = np.array([dist])
            ii = np.array([ii])

        NNdist[k] = dist[0] # save the nearest neighbor distance for entity k
        dist_binomial = dxy * np.abs((centroids[k,:] - tree.data[ii])) # compute the distance to each neighbor in the x and y directions
        size = 2 * np.maximum(dist_binomial[:,0], dist_binomial[:,1])
        size = size[size <= rmax]
        values, counts = np.unique(np.digitize(size, bins=bins, right=True), return_counts=True) # bin the sizes into a histogram based on the specified bins
        valid = values < len(bins)
        hist[values[valid]] = counts[valid]
        cum_hist = np.cumsum(hist) # convert the histogram into a cumulative count of neighbors within each distance bin
        cum_counting[k, :] = cum_hist
        dist = dist[dist < rmax]

    # Mean number of neighbors from a given centroid as a function of box size. 
    # Equivalent to lambda * K(r), where lambda is the spatial density of points and K(r) is the Ripley's function. 
    mean_count = np.mean(cum_counting, axis = 0)

    # Observed Besag's L function as detailed in Biagioli and Tompkins (2023) [eqn. 20 from their paper]. 
    Besag = np.sqrt(mean_count*domain_x*domain_y/(ncnv-1))

    return Besag

In [17]:
def calculate_besag_theor(random_scenes, bins):
    """
    Calculate the 'theoretical' Besag's L function from the randomized iterations of a real radar scene. 
    Note that this is not a ~true~ theoretical Besag's L function. (We treat each randomized scene as an 'observed' scene instead.) 

    Arguments:
    random_scenes: List of 2D arrays representing the randomized radar scenes generated from a real radar scene.
        bins: 1D array of distance bins for calculating the cumulative count of neighbors within each distance bin


    Returns:
        Besag_theor: 1D array representing the average Besag's L function across all randomized radar scenes (the 'theoretical' Besag's L function)
        Besag_list: List of 1D arrays representing Besag's L function for each random iteration
    """
    
    # Create an empty list to store the Besag's L function for each random iteration.
    besag_list = []

    if random_scenes is None or len(random_scenes) == 0:
        return np.zeros_like(bins, dtype=float), []
    else: 
        for scene in random_scenes:
            besag = calculate_besag_for_scene(scene, bins) # calculate Besag's L function for the current random scene as if it were 'observed'
            besag_list.append(besag)
        besag_theor = np.mean(besag_list, axis=0) # find the average Besag's L function across all random iterations to get the 'theoretical' Besag's L function
        return besag_theor, besag_list

#### Calculations

In [18]:
from tqdm import tqdm

In [ ]:
# Rename data variables for easier use in the code below.
dbz = dbz_120km[:50,start_idx:end_idx,start_idx:end_idx] # using the first 50 scenes as an example 
print(dbz.shape)

latitude = latitude[:50,start_idx:end_idx,start_idx:end_idx]
print(latitude.shape)

longitude = longitude[:50,start_idx:end_idx,start_idx:end_idx]
print(longitude.shape)

(50, 241, 241)
(50, 241, 241)
(50, 241, 241)


In [20]:
# Get the shape/mask and a list of precipitating entity areas for each radar scene in the 120 km subset.
results = [
    get_scene_data(dbz[t], data_min=-30, data_max=np.inf)
    for t in tqdm(range(len(dbz)), desc="Create blank masks and finding precipitating entities.", total=len(dbz))
]

masks, areas = zip(*results)
print(len(masks), len(areas))

Create blank masks and finding precipitating entities.: 100%|██████████| 50/50 [00:00<00:00, 178.14it/s]

50 50


In [21]:
# Generate the synthetically randomized radar scenes for each real scene.
results = [
    generate_random(masks[t], areas[t], n_threshold=9, repeat_count=100)
    for t in tqdm(range(len(dbz)), desc="Creating randomized scenes:", total=len(dbz))
]

random_scenes = results
print(len(random_scenes))

Creating randomized scenes:: 100%|██████████| 50/50 [00:09<00:00,  5.20it/s]

50


In [22]:
# Create a list of empty lists to store results.
besag_obs_list = [] # real observed Besag's L function 
besag_theor_list = [] # synthetic 'theoretical' Besag's L function 
l_org_list = [] # Lorg value for each scene
all_besag_theor_list = []  # used for statistical significance testing 

for t in range(len(random_scenes)):

    # Find the number of entities for the scene.
    nentities = len(areas[t])

    # If N < 10, we skip the computation of Lorg. 
    if nentities < 10: 
        besag_obs_list.append(np.full_like(bins, np.nan, dtype=float))
        besag_theor_list.append(np.full_like(bins, np.nan, dtype=float))
        l_org_list.append(np.nan)
        all_besag_theor_list.append([np.full_like(bins, np.nan, dtype=float)])
        continue

    # Raise errors if there are issues. 
    else: 
        try:
            Besag_obs = calculate_besag_for_scene(dbz[t], bins, data_min = - 30, data_max = np.inf)
        except ValueError as e:
            if "data must be finite" in str(e):
                besag_obs_list.append(np.full_like(bins, np.nan, dtype=float))
                besag_theor_list.append(np.full_like(bins, np.nan, dtype=float))
                l_org_list.append(np.nan)
                all_besag_theor_list.append([np.full_like(bins, np.nan, dtype=float)])
                continue
            else:
                raise

        # Continue if no errors are found. 

        # Normalize Besag's L function by rmax.
        Besag_obs=Besag_obs/rmax
        besag_obs_list.append(Besag_obs)
     
        # Calculate the 'theoretical' Besag's L function from the randomized iterations of the real radar scene.
        Besag_theor, Besag_list = calculate_besag_theor(random_scenes[t], bins)
        all_besag_theor_list.append(Besag_list)

        # Normalize the 'theoretical' Besag's L function by rmax.
        Besag_theor=Besag_theor/rmax
        besag_theor_list.append(Besag_theor)
    
        # Compute Lorg as the area between the observed and 'theoretical' Besag's L functions, normalized by rmax.
        L_org = np.trapz(Besag_obs-Besag_theor, x = bins)/rmax
        l_org_list.append(L_org)

        # Optionally print the Lorg value for the current scene to track progress.
        # print(f"Processed time {t+1}/{len(random_scenes)}: L_org = {L_org}")

/tmp/ipykernel_1103011/1144511542.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  L_org = np.trapz(Besag_obs-Besag_theor, x = bins)/rmax


### $I_{org}$ 

$I_{org}$ was defined by Tompkins and Semie 2017[4] at https://doi.org/10.1002/2016MS000802.  

#### Functions

In [23]:
def ecdf(x_inputs):
    """
    Calculate the empirical cumulative distribution function of a set of data points. 

    Arguments: 
        x_inputs: 1D array that the ECDF is calculated from

    Returns:
        y_values: 1D array, the ECDF
        x_values: 1D array, the values ECDF is evaluated at using x_inputs
    """

    import numpy as np

    # Sort the input data points and create an array of x values that span the range of the input data.
    x_sorted = np.sort(x_inputs)
    x_values = np.linspace(start=min(x_sorted),
                           stop=max(x_sorted),
                           num=len(x_sorted))

    # Create a y array of the same shape as x_values to store the ECDF values.
    y_values = np.zeros(np.shape(x_values))

    # Compute the ECDF values.
    for i in range(len(x_values)):
        temp  = x_inputs[x_inputs <= x_values[i]]
        value = np.float64(len(temp))/np.float64(len(x_inputs))
        y_values[i] = value

    return(x_values,y_values)

In [24]:
def calc_IORG(data_array, random_scenes, data_min = -30, data_max = np.inf, n_threshold = 10, includeextras=False):
    """
    Calculate the Iorg value for a given oberved radar scene. 

    Arguments: 
        data_array: 2D array of radar scene data (e.g., rain rate, reflectivity)
        random_scenes: List of 2D arrays representing the randomized radar scenes. 
        data_min : Minimum data value threshold for identifying precipitating entities (default: -30 dBZ)
        data_max : Maximum data value threshold for identifying precipitating entities (default: infinity)
        n_threshold : Minimum number of entities required to calculate Iorg (default: 10)

    Returns:
        iorg_final: Iorg metric for the observed radar scene. 
        If includeextras is True, also returns:
            iorg_values: List of Iorg values for each random iteration (before averaging to get the final Iorg value for the scene)
            real_cdfs: List of CDFs of the nearest neighbor distances in the real radar scene (interpolated onto a common x-axis)
            theor_cdfs: List of CDFs of the nearest neighbor distances in each synthetic random scene (interpolated onto a common x-axis)
            nnd_real: 1D array of nearest neighbor distances for the real radar scene
            nnd_random_list: List of 1D arrays of nearest neighbor distances for each synthetic random scene
            all_random_nnds: 1D array of all nearest neighbor distances from all synthetic random scenes combined together

    """

    # Define the entity mask based on the min and max thresholds, while properly handling NaNs and masked values.
    rain_mask = np.where(
        (np.ma.getmask(data_array)) | np.isnan(data_array) | (data_array == '--') | (data_array == -32768),
        np.nan,
        np.where((data_array >= data_min) & (data_array < data_max), 1, 0)
    )

    # Label each 8-connected structure. 
    s = generate_binary_structure(2, 2) 
    entity_mask, nentities = label(np.nan_to_num(rain_mask, nan=0), structure=s)

    # If the n_threshold is not met, return NaN. 
    if nentities < n_threshold:
        if includeextras == True:
            return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
        else:
            return np.nan

    # Find the centroid of each precipitating entity. 
    centroids = center_of_mass(
        np.where(np.isnan(data_array), 0, data_array.filled(0)) if hasattr(data_array, 'filled') else np.nan_to_num(data_array, nan=0),
        entity_mask, range(1, nentities + 1)
    )
    real_centroids = np.array(centroids)


    # Remove any centroids with NaN values if there was an issue with the center_of_mass calculation.
    real_centroids = real_centroids[~np.isnan(real_centroids).any(axis=1)]

    # Double check that there are at least 2 valid centroids to compute nearest neighbor distances. If not, return NaN.
    if len(real_centroids) < 2:
        if includeextras == True:
            return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
        else:
            return np.nan


    # Find the nearest neighbor distances for each of real centroids. 
    classifier = NearestNeighbors(n_neighbors=2)
    classifier.fit(real_centroids)
    m, _ = classifier.kneighbors(real_centroids)
    nnd_real = m[:, 1] # get the second nearest neighbor distances (first one is itself!)
    x_real, y_real = ecdf(nnd_real)


    iorg_values = []
    real_cdfs = [] # cdf of the NNDs in the real radar scene
    theor_cdfs = [] # cdfs of the NNDs in each synthetic random scene
    nnd_random_list = [] # all NNDs from EACH random scene (nested list)
    all_random_nnds = [] # all the random scene NNDs combined together in a single list


    # Loop through all of the synthetically random scenes to compute Iorg. 
    for t in range(len(random_scenes)):

            # Get the randomized data array.
            rand_array = random_scenes[t]

            # Find the entities and their centroids in the randomized scene. 
            rand_mask = np.where(np.isnan(rand_array), 0, np.where(rand_array > 0, 1, 0))
            s = generate_binary_structure(2, 2)
            entity_mask, nentities = label(rand_mask, structure=s)
            centroids = center_of_mass(rand_array, entity_mask, range(1,nentities+1))
            random_centroids = np.array(centroids)

            if len(random_centroids) > 1:
                # Find the nearest neighbor distances for each of the random centroids.
                classifier = NearestNeighbors(n_neighbors=2)
                classifier.fit(random_centroids)
                m, _ = classifier.kneighbors(random_centroids)
                nnd_random = m[:, 1]
                nnd_random_list.append(nnd_random)
                x_rand, y_rand = ecdf(nnd_random)
                all_random_nnds.extend(nnd_random)
                x_common = np.linspace(min(x_real.min(), x_rand.min()), max(x_real.max(), x_rand.max()), 200)

                # Interpolate both the real and random CDFs onto the same x-axis.
                interp_real_cdf = np.interp(x_common, x_real, y_real)
                interp_real_cdf[0] = 0
                interp_real_cdf[-1] = 1
                real_cdfs.append(interp_real_cdf)

                interp_rand_cdf = np.interp(x_common, x_rand, y_rand)

                # Make sure integration goes from 0 to 1. 
                interp_rand_cdf[0] = 0 
                interp_rand_cdf[-1] = 1
                
                theor_cdfs.append(interp_rand_cdf)

                # Compute Iorg as the area between the real and random CDFs using the trapezoidal rule.
                iorg = np.trapezoid(interp_real_cdf, interp_rand_cdf)
                iorg_values.append(iorg)

                # Mean of all Iorg values across each random iteration is the final Iorg value for the scene.
                iorg_final = np.nanmean(iorg_values)

    if includeextras == True:
        return iorg_final, iorg_values, real_cdfs, theor_cdfs, nnd_real, nnd_random_list, all_random_nnds

    else: 
        return iorg_final

In [25]:
# Compute Iorg for each scene in the 120km subset. 
results = [
    calc_IORG(dbz[t], random_scenes[t], includeextras=True)
    for t in tqdm(range(len(dbz)), desc="Computing Iorg:", total=len(dbz))
]

iorg_list, iorg_values, real_cdfs, theor_cdfs, nnd_real, nnd_random_list, all_random_nnds  = zip(*results)

Computing Iorg:: 100%|██████████| 50/50 [00:06<00:00,  8.31it/s]


### References

[1] Retsch MH, Jakob C, Singh M. 2020. Assessing Convective Organization in Tropical Radar Observations. Journal of Geophysical Research: Atmospheres 125(7), doi:10.1029/2019jd031801, URL http://dx.doi.org/10.1029/2019JD031801.

[2] Tobin I, Bony S, Roca R. 2012. Observational Evidence for Relationships between the Degree of Aggregation of Deep Convection, Water Vapor, Surface Fluxes, and Radiation. Journal of Climate 25(20), doi: 10.1175/jcli-d-11-00258.1, URL http://dx.doi.org/10.1175/JCLI-D-11-00258.1.

[3] Biagioli G, Tompkins AM. 2023. Measuring Convective Organization. Journal of the Atmospheric Sciences 80(12), doi:10.1175/jas-d-23-0103.1, URL
http://dx.doi.org/10.1175/JAS-D-23-0103.1.

[4] Tompkins AM, Semie AG. 2017. Organization of tropical convection in low vertical wind shears: Role of updraft entrainment. Journal of Advances in Modeling Earth Systems 9(2), doi:10.1002/2016ms000802, URL http://dx.doi.org/10.1002/2016MS000802. 
